# pymodal — End-to-end demo

Covers the full pipeline from synthetic structural-health data to a trained PyTorch
classifier, exercising every major API surface of the **signal** branch.

| Section | pymodal feature |
|---|---|
| 1 | Synthetic SDOF accelerance FRFs |
| 2 | `frf` — single-signal API, plotting, processing |
| 3 | `timeseries` — time-domain API, processing, collection, augmentation |
| 4 | `frf_collection` — batch operations, HDF5 persistence |
| 5 | `HDF5Dataset` + PyTorch `DataLoader` |
| 6–8 | 1-D CNN training with gradient accumulation and mixed precision |

> **Environment**: `numpy`, `scipy`, `matplotlib`, `pint`, `pyFRF`, `h5py`,
> `audiomentations`, `torch` (all declared in `setup.py`).  
> `scikit-learn` is **not** required; the confusion matrix is built manually.

## 0 · Imports and device

In [ ]:
import warnings
warnings.filterwarnings("ignore")   # suppress pint unit-stripping notices

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import fftconvolve
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import pymodal
from pymodal import frf, frf_collection, timeseries, timeseries_collection

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RNG = np.random.default_rng(42)

print(f"Device  : {DEVICE}")
print(f"PyTorch : {torch.__version__}")

## 1 · Synthetic data — SDOF accelerance FRFs

Three structural health states are simulated with 60 realisations each.  
Within each class, independent ±3 % scatter is applied to mass and stiffness
to mimic unit-to-unit manufacturing variability.

| Class | Label | Change |
|---|---|---|
| Healthy | 0 | — |
| Cracked | 1 | −10 % stiffness, +75 % damping |
| Loose   | 2 | +10 % mass |

Analytical accelerance:  
$$H(\omega) = \frac{-\omega^2}{k - m\omega^2 + j c\omega}$$

In [ ]:
# ── Signal parameters ─────────────────────────────────────────────────────────
FS  = 100.0             # sampling frequency [Hz]
DT  = 1.0 / FS          # time step [s]  (pass as time_step= to timeseries)
T   = 10.0              # record duration [s]
t   = np.arange(0, T, DT)
N   = len(t)

# One-sided frequency grid (FFT of a real signal)
DF    = 1.0 / T                            # resolution [Hz]
freqs = np.arange(0, FS / 2 + DF, DF)     # 0 … 50 Hz, 501 bins
omega = 2 * np.pi * freqs

print(f"Time  : {N} samples @ {FS} Hz  →  {T} s")
print(f"Freq  : {len(freqs)} bins,  0–{freqs[-1]:.0f} Hz,  Δf = {DF} Hz")

In [ ]:
CLASS_DEFS = [
    {"label": 0, "tag": "Healthy", "m": 1.00, "k": 400.0, "zeta": 0.020},
    {"label": 1, "tag": "Cracked", "m": 1.00, "k": 360.0, "zeta": 0.035},
    {"label": 2, "tag": "Loose",   "m": 1.10, "k": 400.0, "zeta": 0.020},
]
N_PER_CLASS = 60


def sdof_accelerance(omega, m, k, zeta):
    """Analytical SDOF accelerance FRF: ẍ/F = −ω² / (k − mω² + jcω)."""
    c = 2 * zeta * np.sqrt(k * m)
    return -omega**2 / (k - m * omega**2 + 1j * c * omega)


H_all, labels_all, names_all = [], [], []

for cls in CLASS_DEFS:
    for i in range(N_PER_CLASS):
        # Independent ±3 % scatter on mass and stiffness
        mk = RNG.uniform(0.97, 1.03)
        kk = RNG.uniform(0.97, 1.03)
        H = sdof_accelerance(omega, cls["m"] * mk, cls["k"] * kk, cls["zeta"])
        H_all.append(H)
        labels_all.append(float(cls["label"]))
        names_all.append(f"{cls['tag']}_{i:03d}")

print(f"Generated {len(H_all)} FRFs  ({N_PER_CLASS} per class)")

## 2 · Single-signal API — `frf`

`frf` stores a frequency-domain measurement with full unit awareness via
[pint](https://pint.readthedocs.io/).  Measurements are always
`(n_freq, n_outputs, n_inputs)` complex arrays.

In [ ]:
# Create one representative frf object per class (used for single-signal demos)
demo_frfs = [
    frf(
        measurements=H_all[cls["label"] * N_PER_CLASS][:, np.newaxis, np.newaxis],
        freq_resolution=DF,
        measurements_units="millimeter / second**2 / newton",
        freq_units="hertz",
        method="SIMO",
        name=cls["tag"],
    )
    for cls in CLASS_DEFS
]

f0 = demo_frfs[0]
print(f"Measurements shape : {f0.measurements.shape}")
print(f"Frequency range    : {f0.freq_start} – {f0.freq_end}")
print(f"Resolution         : {f0.freq_resolution}")
print(f"DOF                : {f0.dof}")
print(f"Units              : {f0.measurements_units}")

In [ ]:
# pymodal's plot() method supports multiple formats
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
for f_obj, cls in zip(demo_frfs, CLASS_DEFS):
    freq = f_obj.freq_array.magnitude
    mag  = np.abs(f_obj.measurements[:, 0, 0].magnitude)
    ph   = np.angle(f_obj.measurements[:, 0, 0].magnitude)
    axes[0].semilogy(freq, mag,  label=cls["tag"])
    axes[1].plot    (freq, ph,   label=cls["tag"])
axes[0].set(ylabel="|H| (mm/s²/N)", title="SDOF Accelerance — three structural states")
axes[1].set(ylabel="Phase (rad)", xlabel="Frequency (Hz)")
axes[1].set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi],
                   [r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"])
for ax in axes:
    ax.legend()
    ax.grid(True, linestyle=":")
plt.tight_layout()
plt.show()

In [ ]:
# ── change_freq_span : crop to a sub-band ──────────────────────────────────────
f_trimmed = demo_frfs[0].change_freq_span(new_max_freq=20.0)
print(f"Original : {len(demo_frfs[0])} lines  (0–50 Hz)")
print(f"Trimmed  : {len(f_trimmed)} lines  (0–20 Hz)")

# ── change_freq_resolution : coarsen via interpolation ────────────────────────
f_coarse = demo_frfs[0].change_freq_resolution(new_resolution=0.5)
print(f"Coarsened: {len(f_coarse)} lines  Δf = {f_coarse.freq_resolution}")

# Both operations return deep copies; the original is unchanged
assert len(demo_frfs[0]) == 501

## 3 · Time-domain API — `timeseries`

`timeseries` stores a time-domain measurement.  Below we simulate an SDOF
acceleration response to a unit impulse using modal superposition, then
demonstrate the collection and augmentation API.

> **API**: pass `time_step=DT` (Δt in seconds) to the constructor.
> `ts.time_step` returns Δt; `ts.sampling_rate` returns fs = 1/Δt in Hz.
> `change_sampling_rate(new_sampling_rate)` takes the new fs in Hz.

In [ ]:
m, k, zeta = 1.0, 400.0, 0.02
omega_n = np.sqrt(k / m)
omega_d = omega_n * np.sqrt(1 - zeta**2)

# SDOF displacement impulse response h(t)
h_disp = np.exp(-zeta * omega_n * t) * np.sin(omega_d * t) / (m * omega_d)

# Unit-impulse excitation
exc_arr       = np.zeros(N)
exc_arr[0]    = 1.0

# Displacement response (convolution), then acceleration via finite difference
resp_disp = fftconvolve(exc_arr, h_disp, mode="full")[:N] * DT
resp_acc  = np.gradient(np.gradient(resp_disp, DT), DT)

print(f"Response range: [{resp_acc.min():.3f}, {resp_acc.max():.3f}] m/s²")

In [ ]:
# Create pymodal timeseries objects
# time_step = DT = 0.01 s;  sampling_rate = 1/DT = 100 Hz (derived)
ts_resp = timeseries(
    measurements=resp_acc,
    time_step=DT,
    measurements_units="meter / second**2",
    method="SIMO",
    name="SDOF_accel",
)
ts_exc = timeseries(
    measurements=exc_arr,
    time_step=DT,
    measurements_units="newton",
    method="excitation",
    name="SDOF_force",
)

print(f"Duration      : {ts_resp.time_span}")
print(f"Samples       : {len(ts_resp)}")
print(f"time_step     : {ts_resp.time_step}  (Δt in seconds)")
print(f"sampling_rate : {ts_resp.sampling_rate}  (fs in Hz)")

ax, _ = ts_resp.plot(title="SDOF Acceleration Response (impulse excitation)")
plt.show()

In [ ]:
# change_time_span — crop to first 5 s
ts_trimmed = ts_resp.change_time_span(new_max_time=5.0)
print(f"Original  : {len(ts_resp)} samples  ({ts_resp.time_span})")
print(f"Trimmed   : {len(ts_trimmed)} samples  ({ts_trimmed.time_span})")

# change_sampling_rate — halve to 50 Hz (new_sampling_rate is fs in Hz)
ts_half = ts_resp.change_sampling_rate(new_sampling_rate=FS / 2)
print(f"Half rate : {len(ts_half)} samples  fs={ts_half.sampling_rate} Hz  Δt={ts_half.time_step} s")

In [ ]:
# Build a timeseries_collection from 6 noisy realisations
ts_variants = [
    timeseries(
        measurements=resp_acc + RNG.normal(0, 0.02, N),
        time_step=DT,
        measurements_units="meter / second**2",
        method="SIMO",
        name=f"run_{i:02d}",
    )
    for i in range(6)
]

ts_col = timeseries_collection(ts_variants, labels=[0.0] * 6, path="demo_ts.h5")
print(f"Before augmentation: {len(ts_col)} signals")

# AddGaussianNoise appends augmented copies in-place inside the HDF5 file
ts_col.AddGaussianNoise(min_amplitude=0.005, max_amplitude=0.02)
print(f"After  augmentation: {len(ts_col)} signals")

ts_col.close(keep=False)   # discard temporary file
print("Temporary HDF5 removed.")

## 4 · FRF collection — `frf_collection`

`frf_collection` writes every signal array directly into an HDF5 file on
construction.  Arrays are stored as **complex64** (halved storage vs. complex128).
All 180 FRFs (180 × 501 × 8 bytes ≈ 0.7 MB) live on disk; nothing is held in
RAM except the open file handle and HDF5 dataset references.

In [ ]:
COLL_PATH = Path("demo_frfs.h5")

# Wrap every synthesised FRF in a pymodal frf object
# Note: the collection's __init__ nullifies all attributes on frf_objects[0]
# (it becomes collection.collection_class, a zeroed-out template).
# Do not use frf_objects[0] directly after this call.
frf_objects = [
    frf(
        measurements=H[:, np.newaxis, np.newaxis],
        freq_resolution=DF,
        measurements_units="millimeter / second**2 / newton",
        freq_units="hertz",
        method="SIMO",
        name=name,
    )
    for H, name in zip(H_all, names_all)
]

collection = frf_collection(frf_objects, labels=labels_all, path=COLL_PATH)
print(f"Collection : {len(collection)} FRFs  →  {COLL_PATH}")
print(f"Labels     : {sorted({l[()] for l in collection.labels})}")

In [ ]:
# Batch-restrict every FRF to 0–25 Hz in-place (streams through HDF5)
collection.change_freq_span(new_max_freq=25.0)
n_freq_trimmed = collection.measurements[0].shape[0]
print(f"After change_freq_span: {n_freq_trimmed} frequency lines  (0–25 Hz)")

In [ ]:
# Collection plot: all 180 FRFs coloured by rainbow (magnitude)
ax, _ = collection.plot()
ax.set_title("All 180 FRFs — magnitude (0–25 Hz)")
plt.show()

## 5 · PyTorch dataset and DataLoader

`.torch_dataset()` closes the HDF5 file and wraps it in `HDF5Dataset`,
a `torch.utils.data.Dataset` that lazy-loads individual samples on demand.
Each `__getitem__` opens and closes its own file handle, making it safe for
`DataLoader(num_workers > 0)`.

**Complex → float transform**  
FRF measurements are stored as **complex64**.  PyTorch CNNs expect `float32`,
so we supply a transform that converts to log-magnitude spectra:
```
(n_freq, n_out, n_in) complex64  →  (n_out×n_in, n_freq) float32
```

In [ ]:
def frf_to_tensor(x: np.ndarray) -> torch.Tensor:
    """complex128 (n_freq, n_out, n_in) → float32 (n_out*n_in, n_freq).
    
    Log-magnitude compresses the dynamic range and matches how FRF data is
    typically presented in structural health monitoring.
    """
    mag = np.abs(x).astype(np.float32)    # (n_freq, n_out, n_in)
    mag = mag.reshape(mag.shape[0], -1).T  # (n_out*n_in, n_freq)
    return torch.from_numpy(np.log1p(mag))

In [ ]:
# Close the HDF5 file and wrap it as a PyTorch Dataset
collection.torch_dataset()
dataset = collection.dataset
dataset.transform = frf_to_tensor

x0, y0 = dataset[0]
print(f"Dataset size   : {len(dataset)}")
print(f"Sample shape   : {x0.shape}   (channels, freq_lines)")
print(f"Label          : {y0}  dtype={y0.dtype}")

In [ ]:
# 80 / 20 train-val split
n_total = len(dataset)
n_train = int(0.8 * n_total)
n_val   = n_total - n_train

gen = torch.Generator().manual_seed(42)
train_ds, val_ds = torch.utils.data.random_split(dataset, [n_train, n_val],
                                                  generator=gen)

# num_workers > 0 is safe: HDF5Dataset opens a fresh file handle per __getitem__.
# Use num_workers=0 only on Windows where fork-based multiprocessing is unavailable.
BATCH = 4
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)

xb, yb = next(iter(train_loader))
print(f"Batch shape : {xb.shape}  labels: {yb.tolist()}")
print(f"Train batches: {len(train_loader)}, val batches: {len(val_loader)}")

## 6 · Model — lightweight 1-D CNN

FRF magnitude spectra are treated as 1-D sequences.  Three convolutional
layers extract local spectral features; global average pooling collapses
frequency into a fixed-length embedding before the linear classifier.

In [ ]:
class FRF_CNN(nn.Module):
    """1-D CNN for FRF-based structural health classification."""

    def __init__(self, n_channels: int, n_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(n_channels, 16, kernel_size=7, padding=3), nn.BatchNorm1d(16), nn.ReLU(),
            nn.Conv1d(16,         32, kernel_size=5, padding=2), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32,         64, kernel_size=3, padding=1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Linear(64, n_classes)

    def forward(self, x):
        return self.classifier(self.features(x).squeeze(-1))


N_CHANNELS = x0.shape[0]   # n_out * n_in  (= 1 for SIMO, 1 DOF)
N_CLASSES  = 3

model = FRF_CNN(N_CHANNELS, N_CLASSES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters : {n_params:,}")
print(model)

## 7 · Training — gradient accumulation + mixed precision

Two techniques address the 4 GB VRAM constraint:

| Technique | Benefit | Implementation |
|---|---|---|
| **Gradient accumulation** | Effective batch size × `ACCUM` without extra VRAM | Scale loss, skip `step()` until Nth micro-batch |
| **Mixed precision (AMP)** | ~50 % VRAM reduction, faster matmuls on Ampere+ | `autocast` + `GradScaler` |

With `BATCH=4` and `ACCUM=4`, the effective batch size is **16** while only
4 samples occupy the GPU at once.

In [ ]:
EPOCHS      = 40
LR          = 1e-3
ACCUM       = 4          # gradient accumulation steps
AMP_ENABLED = DEVICE.type == "cuda"

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

train_losses, val_accs = [], []

for epoch in range(1, EPOCHS + 1):
    # ── Training ──────────────────────────────────────────────────────────────
    model.train()
    optimizer.zero_grad()
    epoch_loss = 0.0

    for step, (x, y) in enumerate(train_loader, 1):
        x = x.to(DEVICE)
        y = y.long().to(DEVICE)

        with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
            loss = criterion(model(x), y) / ACCUM   # scale before accumulation

        scaler.scale(loss).backward()

        if step % ACCUM == 0 or step == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        epoch_loss += loss.item() * ACCUM

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

    # ── Validation ────────────────────────────────────────────────────────────
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.long().to(DEVICE)
            with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
                preds = model(x).argmax(1)
            correct += (preds == y).sum().item()

    val_acc = correct / n_val
    val_accs.append(val_acc)
    scheduler.step()

    if epoch % 8 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS}  loss={avg_loss:.4f}  val_acc={val_acc:.3f}")

print("\nTraining complete.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(range(1, EPOCHS + 1), train_losses)
ax1.set(title="Training loss", xlabel="Epoch", ylabel="Cross-entropy")
ax2.plot(range(1, EPOCHS + 1), val_accs)
ax2.set(title="Validation accuracy", xlabel="Epoch", ylabel="Accuracy",
        ylim=(0, 1.05))
ax2.axhline(1.0, color="grey", linestyle=":", linewidth=0.8)
for ax in (ax1, ax2):
    ax.grid(True, linestyle=":")
plt.tight_layout()
plt.show()

## 8 · Evaluation — confusion matrix

In [ ]:
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for x, y in val_loader:
        x = x.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
            preds = model(x).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_true.extend(y.tolist())

CLASS_NAMES = [cls["tag"] for cls in CLASS_DEFS]
n_cls = len(CLASS_NAMES)

# Manual confusion matrix (no sklearn required)
cm = np.zeros((n_cls, n_cls), dtype=int)
for t, p in zip(all_true, all_preds):
    cm[int(t), int(p)] += 1

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(n_cls)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right")
ax.set_yticks(range(n_cls)); ax.set_yticklabels(CLASS_NAMES)
ax.set(xlabel="Predicted", ylabel="True", title="Confusion matrix (validation)")
for i in range(n_cls):
    for j in range(n_cls):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

accuracy = np.trace(cm) / cm.sum()
print(f"Validation accuracy : {accuracy:.1%}")

## 9 · Cleanup

In [ ]:
COLL_PATH.unlink(missing_ok=True)
print(f"Removed {COLL_PATH}")